# LTFSR-Meta - Run ALL four methods in one go (Kaggle-ready)

Companion to `run_experiment.ipynb`. That notebook runs **one** method; this
one loops over **all four** (`baseline`, `prototype`, `contrastive`, `meta`)
in a single online run - training, inference, logging and plotting each - then
builds a side-by-side **comparison table and chart** at the end.

Same `src/` modules, same per-method outputs as the single-method notebook;
the only new thing is the `for` loop and the final comparison.

**On Kaggle:** set `DATA_DIR` / `OUTPUT_DIR` in cell 2, enable GPU, then Run All.

## 0. (Optional) Kaggle-safe dependency setup

**On Kaggle, leave this disabled / skip it.** Kaggle already ships
`torch`, `torchvision`, `numpy`, `pandas`, `scikit-learn`, `matplotlib` and
`Pillow` - everything this project imports. Reinstalling `torch`/`torchvision`
on top of the preinstalled CUDA build causes
`RuntimeError: duplicate registrations for aten.linspace` at `import torch`.

This cell therefore **never reinstalls torch/torchvision if they already exist**
(the Kaggle case). It is only useful when running the notebook somewhere that
lacks them. If you hit the duplicate-registration error, do **Run -> Factory
reset** and run again *without* reinstalling torch.

In [ ]:
# Kaggle-safe: never reinstall torch/torchvision (keeps the preinstalled CUDA build).
import importlib.util
import subprocess
import sys

INSTALL_DEPS = False   # set True only when running OFF Kaggle in a bare environment

if INSTALL_DEPS:
    if importlib.util.find_spec("torch") is None:        # absent only off-Kaggle
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "torch>=2.0.0", "torchvision>=0.15.0"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "scikit-learn>=1.3.0", "matplotlib>=3.7.0",
                    "pandas>=1.5.0", "numpy>=1.23.0", "Pillow>=9.0.0"], check=True)
    print("Dependencies ensured.")
else:
    print("Skipping pip install (expected on Kaggle - deps are preinstalled).")

## 1. Make `src/` importable (works locally and on Kaggle)

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Return the folder containing `src/` (search cwd, parents, Kaggle inputs)."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Configuration - the only cell you normally edit

`METHODS` is the list looped over. Drop any you don't want, or reorder them.
Each method still writes to its own `OUTPUT_DIR/<method>/` folder, so nothing
is overwritten and you can re-run a single method later if needed.

On Kaggle, point `DATA_DIR` at your uploaded dataset (e.g.
`/kaggle/input/cifar-100-lt/CIFAR-100-LT`) and `OUTPUT_DIR` at `/kaggle/working`.

In [ ]:
# --- which methods to run (the loop iterates over this list) ---
METHODS = ["baseline", "prototype", "contrastive", "meta"]

# --- paths ---
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"   # folder with train/ test/ class_counts.json
OUTPUT_DIR = PROJECT_DIR / "outputs"               # where run folders are written
BUILD_DATASET = False   # set True on Kaggle to build CIFAR-100-LT into OUTPUT_DIR first

# --- core hyperparameters (shared by all methods) ---
NUM_CLASSES = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.1     # baseline/prototype/probe use SGD; meta uses META_LR (Adam)
EPOCHS = 100
SEED = 42
PRETRAINED = True       # ImageNet-pretrained ResNet-18 backbone
NUM_WORKERS = 2
DEVICE = None           # None = auto (CUDA if available), or "cpu" / "cuda"

# --- quick smoke test (None = use the full dataset) ---
# Tip: set these to e.g. 2000 / 1000 and EPOCHS to 2 for a fast end-to-end check.
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

# --- contrastive (Method 3) ---
PRETRAIN_EPOCHS = 100
PROBE_EPOCHS = 30
TEMPERATURE = 0.07

# --- few-shot / meta-learning (Method 4) ---
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
EPISODES_PER_EPOCH = 100
META_LR = 0.001

## 3. Imports, reproducibility, device

In [ ]:
import random
import time
import traceback

import numpy as np
import pandas as pd
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    make_loader, split_shot_groups)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.trainers.engine import evaluate
from src.utils.experiment import create_run_dir, save_config, save_history, save_metrics
from src.utils.seed import resolve_device, set_seed

device = resolve_device(DEVICE)
print("Device:", device, "| Methods:", METHODS)

## 4. (Optional) Build the dataset, then load it once and share across methods

In [ ]:
if BUILD_DATASET:
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_DIR / "data" / "prepare_datasets.py"),
                    "--dataset", "cifar100-lt",
                    "--data_dir", str(OUTPUT_DIR), "--overwrite"], check=True)
    DATA_DIR = OUTPUT_DIR / "CIFAR-100-LT"

assert (DATA_DIR / "train").exists(), f"No train/ folder under {DATA_DIR}"
class_counts = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts)
print("Classes per group:", {name: len(ids) for name, ids in shot_groups.items()})
print("Head class count:", max(class_counts.values()), "| Tail class count:", min(class_counts.values()))

In [ ]:
def subsample(dataset, max_samples, seed=SEED):
    """Keep a random subset of an ImageFolder in place (smoke testing only)."""
    if max_samples is None or max_samples >= len(dataset.samples):
        return dataset
    rng = random.Random(seed)
    keep = rng.sample(range(len(dataset.samples)), max_samples)
    dataset.samples = [dataset.samples[i] for i in keep]
    dataset.targets = [dataset.targets[i] for i in keep]
    return dataset


# Three views of the data, loaded once and reused by every method:
#   train_aug  - augmented train set (for learning)
#   train_eval - clean train set (for prototypes / t-SNE / meta support sets)
#   test_eval  - balanced test set (evaluation, the standard LT protocol)
train_aug = subsample(load_split(DATA_DIR, "train", build_transforms(train=True)), MAX_TRAIN_SAMPLES)
train_eval = subsample(load_split(DATA_DIR, "train", build_transforms(train=False)), MAX_TRAIN_SAMPLES)
test_eval = subsample(load_split(DATA_DIR, "test", build_transforms(train=False)), MAX_TEST_SAMPLES)

train_loader = make_loader(train_aug, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"train={len(train_aug)} images | test={len(test_eval)} images")

## 5. Look at the data (long-tail profile, shared by all methods)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
visualize.plot_class_frequency(class_counts, OUTPUT_DIR)
visualize.plot_shot_distribution(shot_groups, OUTPUT_DIR)
print("Saved class_frequency.png and shot_distribution.png to", OUTPUT_DIR)

## 6. The loop - train + evaluate + log + plot each method

`run_one_method` is exactly the per-method logic from `run_experiment.ipynb`,
wrapped in a function. The `for` loop calls it once per method. Each method is
wrapped in `try/except` so a failure in one (e.g. out-of-memory on `contrastive`)
is reported but does **not** stop the others - you still get results for the rest.

In [ ]:
def run_one_method(method: str):
    """Train, evaluate, log and plot a single method. Returns its metrics dict."""
    # Re-seed before every method so each starts from identical, reproducible state.
    set_seed(SEED)
    run_dir = create_run_dir(OUTPUT_DIR, run_name=method)
    print(f"\n{'=' * 70}\n>>> METHOD: {method}  |  run dir: {run_dir}\n{'=' * 70}")
    pretrain_history = None

    if method == "baseline":
        from src.trainers.baseline_trainer import train_baseline
        model, history = train_baseline(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                         epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
        result = evaluate(model, test_loader, device, collect_features=True)

    elif method == "prototype":
        from src.trainers.prototype_trainer import train_prototype
        model, history = train_prototype(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                         epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
        result = evaluate(model, test_loader, device, collect_features=True)

    elif method == "contrastive":
        from src.trainers.contrastive_trainer import train_contrastive
        model, history, pretrain_history = train_contrastive(
            DATA_DIR, train_loader, test_loader, NUM_CLASSES, device, run_dir,
            pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=PROBE_EPOCHS,
            batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
            pretrain_lr=LEARNING_RATE, probe_lr=LEARNING_RATE,
            temperature=TEMPERATURE, pretrained=PRETRAINED)
        result = evaluate(model, test_loader, device, collect_features=True)

    elif method == "meta":
        from src.trainers.meta_trainer import evaluate_meta, train_meta
        encoder, history = train_meta(train_aug, train_eval, device, run_dir,
                                      epochs=EPOCHS, episodes_per_epoch=EPISODES_PER_EPOCH,
                                      n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
                                      learning_rate=META_LR, pretrained=PRETRAINED, seed=SEED)
        result = evaluate_meta(encoder, train_eval_loader, test_loader, NUM_CLASSES, device)

    else:
        raise ValueError(f"Unknown method: {method!r}")

    # --- metrics + log ---
    metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
    print(format_metrics(metrics))

    config = dict(
        method=method, num_classes=NUM_CLASSES, batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE, epochs=EPOCHS, seed=SEED, pretrained=PRETRAINED,
        device=str(device), data_dir=str(DATA_DIR),
        n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY, episodes_per_epoch=EPISODES_PER_EPOCH,
        pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=PROBE_EPOCHS, temperature=TEMPERATURE,
    )
    save_config(run_dir, config)
    save_metrics(run_dir, metrics)
    save_history(run_dir, history)
    if pretrain_history is not None:
        pretrain_history.to_csv(run_dir / "pretrain_metrics.csv", index=False)

    # --- plots ---
    visualize.plot_training_curves(history, run_dir)
    visualize.plot_confusion_matrices(result["y_true"], result["y_pred"], NUM_CLASSES, run_dir)
    if "features" in result:
        visualize.plot_tsne(result["features"], result["y_true"], run_dir)
    print("Saved config / metrics / history / figures to", run_dir)
    return metrics, history


all_metrics = {}
all_history = {}   # per-method per-epoch DataFrames, for the overlay curves
for method in METHODS:
    start = time.time()
    try:
        all_metrics[method], all_history[method] = run_one_method(method)
        print(f"[{method}] done in {time.time() - start:.0f}s")
    except Exception:
        print(f"[{method}] FAILED after {time.time() - start:.0f}s - skipping. Traceback:")
        traceback.print_exc()

print("\nFinished. Methods with results:", list(all_metrics))

## 7. Compare the methods

One table and one bar chart across every method that finished. The comparison
files are written to the top of `OUTPUT_DIR` (alongside the per-method folders).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

assert all_metrics, "No method finished successfully - check the log above."

# Rows = methods, columns = metrics. Ordered to match METHODS.
comparison = pd.DataFrame(
    {m: all_metrics[m] for m in METHODS if m in all_metrics}
).T
comparison.index.name = "method"

comparison_csv = OUTPUT_DIR / "comparison.csv"
comparison.to_csv(comparison_csv)
print("Saved", comparison_csv, "\n")

# Headline metrics for long-tail: overall + balanced accuracy, g-mean, and the
# many/medium/few split. Show whatever is present.
headline = [c for c in ["accuracy", "balanced_accuracy", "g_mean", "macro_f1",
                        "many_shot_accuracy", "medium_shot_accuracy", "few_shot_accuracy"]
            if c in comparison.columns]
with pd.option_context("display.float_format", lambda v: f"{v:.4f}"):
    print(comparison[headline].to_string())

comparison

In [ ]:
# Grouped bar chart of the headline metrics across methods.
plot_metrics = [c for c in ["accuracy", "balanced_accuracy", "g_mean",
                            "many_shot_accuracy", "medium_shot_accuracy", "few_shot_accuracy"]
                if c in comparison.columns]
methods = list(comparison.index)
x = np.arange(len(plot_metrics))
width = 0.8 / max(len(methods), 1)

fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(plot_metrics)), 5))
for i, method in enumerate(methods):
    ax.bar(x + i * width, comparison.loc[method, plot_metrics].values, width, label=method)
ax.set_xticks(x + width * (len(methods) - 1) / 2)
ax.set_xticklabels(plot_metrics, rotation=30, ha="right")
ax.set_ylabel("Score")
ax.set_title("Method comparison on CIFAR-100-LT (test set)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()

comparison_png = OUTPUT_DIR / "comparison.png"
fig.savefig(comparison_png, dpi=150)
plt.close(fig)
print("Saved", comparison_png)

## 8. Training / validation curves across methods

All methods on the same axes: one chart for **loss**, one for **accuracy**.
Each method gets its own colour; **solid = train, dashed = validation**.
Methods can have different epoch counts (e.g. the contrastive linear probe runs
`PROBE_EPOCHS`) - the x-axis is just the epoch number, so shorter curves stop early.

In [ ]:
# Reuses the Agg backend / pyplot imported in section 7.
def plot_overlay(histories, train_col, val_col, ylabel, title, fname):
    """Overlay one (train_col, val_col) pair for every method on a single chart."""
    valid = {m: h for m, h in histories.items() if h is not None and len(h)}
    if not valid:
        print("No history to plot for", title)
        return
    cmap = plt.get_cmap("tab10")
    fig, ax = plt.subplots(figsize=(9, 5.5))
    for i, (method, h) in enumerate(valid.items()):
        color = cmap(i % 10)
        if train_col in h:
            ax.plot(h["epoch"], h[train_col], color=color, linestyle="-",
                    label=f"{method} train")
        if val_col in h:
            ax.plot(h["epoch"], h[val_col], color=color, linestyle="--",
                    label=f"{method} val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    path = OUTPUT_DIR / fname
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print("Saved", path)


assert all_history, "No method finished - nothing to plot."

# NOTE: loss magnitudes differ a lot across methods (e.g. prototype uses a
# distance loss in the hundreds while the baseline cross-entropy is ~1-5), so
# on a shared linear axis the largest-loss method dominates. The accuracy chart
# is the fair side-by-side comparison; the loss chart is mainly for convergence shape.
plot_overlay(all_history, "train_loss", "val_loss", "Loss",
             "Training & validation loss across methods", "compare_loss_curves.png")
plot_overlay(all_history, "train_accuracy", "val_accuracy", "Accuracy",
             "Training & validation accuracy across methods", "compare_accuracy_curves.png")

## 9. Summary

After Run All, `OUTPUT_DIR` holds one self-contained folder per method plus the
cross-method comparison:

```
outputs/
+-- class_frequency.png / shot_distribution.png   # shared long-tail profile
+-- comparison.csv / comparison.png               # final metrics, all methods side by side
+-- compare_loss_curves.png                       # train/val loss, 4 methods overlaid
+-- compare_accuracy_curves.png                   # train/val accuracy, 4 methods overlaid
+-- baseline/      config.json metrics.json metrics.csv best_model.pt *.png
+-- prototype/     ...
+-- contrastive/   ... (+ pretrain_metrics.csv)
+-- meta/          ...
```

To re-run just one method, set `METHODS = ["meta"]` and Run All again - only that
folder is rewritten. Compare `g_mean`, `balanced_accuracy` and `few_shot_accuracy`
in `comparison.csv` to see which method handles the tail best.